<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='6.%20databases_with_flask_sqlalchemy.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 6: Databases</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../3.%20application_structure/8.%20blueprints_and_application_factory.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 8: Blueprints →</a>
</div>


# Chapter 7: Database Migrations with Flask-Migrate — Evolving Your Schema

> *"Your database schema will change. The question is whether you'll change it safely or destructively."*


## 🎯 The Big Picture

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

You built a beautiful `User` model with id, username, and email. Then your product manager asks for a `bio` field. Then `profile_picture_url`. Then `is_email_verified`. Then `last_login`.

Without migrations, every schema change means:
1. Drop the entire database
2. Recreate it from scratch
3. **Lose all your data** — users, posts, everything

That's acceptable in development (you have no real data). It's **catastrophic in production**.

**Migrations** are the solution. They're like **Git commits for your database schema** — each change is recorded, reversible, and can be applied incrementally to any database without losing data.

**What you'll learn in this chapter:**
- What migrations are and why they're essential for any real application
- Setting up Flask-Migrate and the `flask db` CLI
- The complete migration workflow: init → edit model → migrate → review → upgrade
- Reading and understanding generated migration scripts
- Rolling back with `flask db downgrade`
- Tracking history with `flask db history` and `flask db current`
- Production migration best practices
- Common pitfalls: data migrations, NOT NULL columns, renames, merge conflicts

> ❓ **Why do I need migrations?** When your data model changes (e.g., you add a column), the database schema must change too. Migrations are version-controlled scripts that apply those changes in order, keeping every developer and your production server in sync.

## 🧠 Core Concepts: The "Why"

The easiest way to read this section is to keep one plain-language question in view: what is The "Why" actually responsible for? Once that job is clear, the terminology stops feeling arbitrary and the details start to line up.

### The Git Analogy

Think of database migrations exactly like Git commits for your schema:

```
Git:                               Migrations:
─────────────────────────────────────────────────────
git init                     ↔    flask db init
git add . && git commit -m   ↔    flask db migrate -m "description"
git log                      ↔    flask db history
git push (deploy)            ↔    flask db upgrade
git revert HEAD              ↔    flask db downgrade
git checkout <hash>          ↔    flask db downgrade <revision>
```

Each migration file = one recorded, reversible schema change.

### What Happens Without Migrations

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

```
Week 1: Python model has  {id, username, email}
        Database table has {id, username, email}   <- in sync

Week 2: You add to model:  bio = db.Column(db.String(500))
        Python model has  {id, username, email, bio}
        Database table has {id, username, email}   <- MISMATCH!

        $ flask run
        -> User.query.all()
        -> sqlalchemy.exc.OperationalError: no such column: user.bio
```

Migrations keep the Python models and database schema **in sync**.
### What Alembic Does Under the Hood

Flask-Migrate is a wrapper around **Alembic** (the SQLAlchemy migration tool):

1. Reads your Python models (`models.py`)
2. Reads the current database schema (introspects the actual DB)
3. **Diffs** them: what changed?
4. Generates a Python script that transforms old schema → new schema
5. Stores the script in `migrations/versions/`
6. You review it, then run `flask db upgrade` to apply it

> ❓ **Why use an ORM instead of raw SQL?** An ORM (Object-Relational Mapper) lets you work with database rows as Python objects, so you write Python instead of SQL strings. It also prevents SQL injection by default and makes switching databases straightforward later.

## ⚡ Syntax & First Usage: Setup

A beginner usually learns this material faster by connecting each setup step to its purpose rather than treating it as a checklist. Keep asking what each piece enables and what would fail if you skipped it.



In [ ]:
# Step 1: Install
# pip install flask-migrate

# Step 2: Wire into your Flask app
setup_code = '''
from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from flask_migrate import Migrate

app = Flask(__name__)
app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///blog.db"

db = SQLAlchemy(app)
migrate = Migrate(app, db)   # <- This one line enables all "flask db" commands
'''
print(setup_code)

print()
print("All flask db commands (available after Migrate is initialized):")
print()
commands = [
    ("flask db init",                "Create migrations/ folder. Run ONCE per project."),
    ("flask db migrate -m 'msg'",   "Auto-generate migration script from model changes."),
    ("flask db upgrade",             "Apply all pending migrations to the database."),
    ("flask db downgrade",           "Revert the most recent migration."),
    ("flask db history",             "List all migrations, oldest to newest."),
    ("flask db current",             "Show which migration the database is currently at."),
    ("flask db heads",               "Show the latest migration revision(s)."),
    ("flask db stamp head",          "Mark DB as up-to-date without running migrations."),
    ("flask db show <rev>",          "Show details of a specific migration."),
    ("flask db merge rev1 rev2",     "Create merge migration to resolve parallel migrations."),
]
for cmd, desc in commands:
    print(f"  {cmd:<35}  {desc}")


## 🔬 Deep Dive: The Complete Migration Workflow

The key to this section is sequence. If you can explain what happens first, what happens next, and where control moves after that, the moving parts here become much easier to reason about.



In [ ]:
print("=== THE COMPLETE MIGRATION WORKFLOW ===")
print()

steps = [
    (
        "STEP 0 — Project Initialization (once per project)",
        "$ flask db init",
        [
            "Creates migrations/ folder and alembic.ini configuration",
            "Run this ONCE when starting a new project",
            "Commit the migrations/ folder to git immediately",
        ]
    ),
    (
        "STEP 1 — Generate first migration",
        "$ flask db migrate -m 'initial migration'",
        [
            "Compares Python models vs empty database",
            "Generates a script that creates all your tables",
            "REVIEW the generated file before proceeding!",
        ]
    ),
    (
        "STEP 2 — Apply the migration",
        "$ flask db upgrade",
        [
            "Runs the upgrade() function in the migration script",
            "Creates the actual database tables",
            "Updates the alembic_version table to track current state",
        ]
    ),
    (
        "STEP 3 — Make a model change",
        "Edit models.py: add bio = db.Column(db.String(500))",
        [
            "Add, remove, or modify a column in your Python model",
            "Don't touch the database yet — just the Python file",
        ]
    ),
    (
        "STEP 4 — Generate migration for the change",
        "$ flask db migrate -m 'add bio column to user'",
        [
            "Diffs your models vs current DB schema",
            "Generates an ALTER TABLE script",
            "ALWAYS review the generated file in migrations/versions/",
        ]
    ),
    (
        "STEP 5 — Apply the change",
        "$ flask db upgrade",
        [
            "Runs upgrade(): adds the 'bio' column to the user table",
            "Existing rows get NULL for bio (since nullable=True)",
            "SAFE: no data is lost",
        ]
    ),
]

for title, command, details in steps:
    print(f"  [{title}]")
    print(f"  Command: {command}")
    for d in details:
        print(f"    -> {d}")
    print()


In [ ]:
# The migrations/ folder structure
print("=== migrations/ folder structure ===")
print()
tree = '''
your_project/
    app.py
    models.py
    migrations/                          <- created by "flask db init"
        alembic.ini                      <- Alembic configuration file
        env.py                           <- migration environment setup
        script.py.mako                   <- template for new migration files
        versions/                        <- one .py file per migration
            abc1_initial_migration.py
            def2_add_bio_to_user.py
            ghi3_add_posts_table.py
            jkl4_add_post_tags.py
'''
print(tree)
print("Key rules:")
print("  - Commit the ENTIRE migrations/ folder to git")
print("  - Other developers run 'flask db upgrade' to sync their local DB")
print("  - Production: run 'flask db upgrade' before restarting the app")
print("  - Add *.db to .gitignore (each developer has their own local DB file)")


### Reading a Migration File — Every Line Explained

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.



In [ ]:
# What a generated migration file looks like
# (Triple-double-quotes in the actual file are shown as comments here)
migration_file = '''
# File: migrations/versions/a3f8b2c1d4e5_add_bio_to_user.py
# Generated by: flask db migrate -m "add bio to user"

# Module docstring would normally be here with:
#   Revision ID: a3f8b2c1d4e5
#   Revises: 9b0e1a2c3d4f     <- parent migration (what came before)
#   Create Date: 2024-01-15 10:30:00

from alembic import op           # op = operations (add/drop/alter columns, tables)
import sqlalchemy as sa          # sa = sqlalchemy type definitions

# Unique ID for this migration (used by "flask db current" and downgrade)
revision = "a3f8b2c1d4e5"

# The revision this migration builds on (forms a linked chain)
down_revision = "9b0e1a2c3d4f"

# Required boilerplate for branching support (usually None)
branch_labels = None
depends_on    = None


def upgrade():
    # ── FORWARD: what to do when applying this migration ──────────
    op.add_column(
        "user",                                          # table name
        sa.Column("bio", sa.String(length=500), nullable=True)  # new column
    )
    op.add_column(
        "user",
        sa.Column("last_login", sa.DateTime(), nullable=True)
    )


def downgrade():
    # ── BACKWARD: how to undo this migration ──────────────────────
    # Columns dropped in REVERSE order of addition
    op.drop_column("user", "last_login")
    op.drop_column("user", "bio")
'''
print(migration_file)


In [ ]:
# Common alembic operations in migration files
print("=== alembic operation reference ===")
print()

ops_list = [
    ("Column operations", [
        "op.add_column('user', sa.Column('bio', sa.String(500), nullable=True))",
        "op.drop_column('user', 'bio')",
        "op.alter_column('user', 'bio', nullable=False, type_=sa.Text())",
        "op.alter_column('user', 'old_name', new_column_name='new_name')",
    ]),
    ("Table operations", [
        "op.create_table('tags', sa.Column('id', sa.Integer, primary_key=True), ...)",
        "op.drop_table('tags')",
        "op.rename_table('old_table', 'new_table')",
    ]),
    ("Index operations", [
        "op.create_index('ix_user_email', 'user', ['email'], unique=True)",
        "op.drop_index('ix_user_email', 'user')",
    ]),
    ("Constraint operations", [
        "op.create_unique_constraint('uq_user_email', 'user', ['email'])",
        "op.drop_constraint('uq_user_email', 'user')",
        "op.create_foreign_key('fk_post_user', 'post', 'user', ['user_id'], ['id'])",
    ]),
    ("Data migration (raw SQL)", [
        "conn = op.get_bind()",
        "conn.execute(sa.text('UPDATE user SET role = :r WHERE role IS NULL'), {'r': 'user'})",
        "op.bulk_insert(user_table, [{'username': 'admin', 'email': 'a@b.com'}])",
    ]),
]

for category, examples in ops_list:
    print(f"  {category}:")
    for ex in examples:
        print(f"    {ex}")
    print()


### Rolling Back — `flask db downgrade`

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:
rollback_code = '''
# ── Check current state ──────────────────────────────────────────
$ flask db current
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
a3f8b2c1d4e5 (head)     <- currently at this revision

# ── See full history ─────────────────────────────────────────────
$ flask db history
<base>         -> 9b0e1a2c3d4f,  initial migration
9b0e1a2c3d4f   -> a3f8b2c1d4e5 (head),  add bio to user

# ── Downgrade one step (undo most recent migration) ──────────────
$ flask db downgrade
INFO  Running downgrade a3f8b2c1d4e5 -> 9b0e1a2c3d4f...
# Result: "bio" and "last_login" columns REMOVED from user table
# Data in those columns is LOST (this is the point of downgrade)

# ── Downgrade to specific revision ──────────────────────────────
$ flask db downgrade 9b0e1a2c3d4f    # jump to exact revision
$ flask db downgrade base             # revert ALL migrations (empty DB)

# ── Re-apply ─────────────────────────────────────────────────────
$ flask db upgrade                    # re-apply all pending migrations
$ flask db upgrade a3f8b2c1d4e5       # upgrade to specific revision

# ── Step counts ─────────────────────────────────────────────────
$ flask db downgrade -1               # go back exactly 1 step
$ flask db downgrade -3               # go back 3 steps
$ flask db upgrade +2                 # go forward 2 steps
'''
print(rollback_code)


### ⚖️ `db.create_all()` vs Flask-Migrate

The hard part here is usually not the syntax but the boundary between similar ideas. Keep comparing the job each option does best, the tradeoff it introduces, and the clues that tell you which one fits the situation in front of you.



In [ ]:
comparison_lines = [
    "+------------------------------+--------------------+---------------------------+",
    "| Feature                      | db.create_all()    | Flask-Migrate             |",
    "+------------------------------+--------------------+---------------------------+",
    "| Creates new tables           | Yes (if missing)   | Yes                       |",
    "| Updates existing tables      | NO (ignores them!) | Yes (ALTER TABLE)         |",
    "| Destroys existing data       | No (skips existing)| No (preserves all data)   |",
    "| Tracks schema history        | No                 | Yes (versioned .py files) |",
    "| Rollback support             | No                 | Yes (downgrade command)   |",
    "| Works for team collaboration | Breaks             | Works (commit migrations) |",
    "| Safe for production          | No                 | Yes (standard practice)   |",
    "| Setup time                   | Zero               | 2 minutes                 |",
    "| Use case                     | Tests / greenfield | Every real project        |",
    "+------------------------------+--------------------+---------------------------+",
    "",
    "Rule of thumb:",
    "  db.create_all()  ->  Unit tests (fresh in-memory DB each run)",
    "                   ->  Very first setup of a brand-new project",
    "  Flask-Migrate    ->  Anything with real data or a team",
]
for line in comparison_lines:
    print(line)


### Production Migration Best Practices

The important beginner shift here is moving from “it works on my machine” to “it behaves predictably in a real environment.” Each step matters because it reduces surprises when the software runs elsewhere.



In [ ]:
print('''
=== PRODUCTION MIGRATION CHECKLIST ===

BEFORE touching production:
  1. BACKUP the production database!
       PostgreSQL: $ pg_dump mydb > backup_$(date +%Y%m%d_%H%M).sql
       SQLite:     $ cp prod.db prod_backup_$(date +%Y%m%d).db
       MySQL:      $ mysqldump mydb > backup.sql

  2. Test the migration on a STAGING environment first
       $ FLASK_ENV=staging flask db upgrade
       (Catches issues before they hit real users)

  3. Review the migration file manually
       - Does upgrade() do exactly what you intend?
       - Does downgrade() correctly reverse it?
       - Could any step fail halfway through?

SAFE migrations (can apply without downtime):
  + Adding a nullable column
  + Adding a new table
  + Adding an index (use CONCURRENTLY in PostgreSQL)
  + Dropping an unused index

DANGEROUS migrations (require careful planning):
  ! Adding a NOT NULL column without server_default
    -> Will fail if the table has existing rows
    -> Solution: add as nullable, populate data, then add NOT NULL

  ! Dropping a column that code still references
    -> Deploy code change first, THEN drop the column
    -> "Expand/Contract" deployment pattern

  ! Renaming a column
    -> Auto-generate sees DROP + ADD = DATA LOSS!
    -> Use op.alter_column(..., new_column_name=...) explicitly

  ! Large table migrations (millions of rows)
    -> ALTER TABLE locks the table — users see errors
    -> Use pg_repack (PostgreSQL) or pt-online-schema-change (MySQL)

PRODUCTION DEPLOYMENT ORDER:
  $ git pull                   # get new code with migration files
  $ flask db upgrade           # apply migrations to production DB
  $ systemctl restart myapp    # restart the application server
''')


### Data Migrations — When Auto-Generate Isn't Enough

If this section feels dense, pause and identify the data structure or model first. Most of the APIs here make more sense once you know what is being stored, queried, or transformed.



In [ ]:
data_migration = '''
# Scenario: Split "full_name" into "first_name" + "last_name"
# Auto-generate cannot do this — it requires manual data transformation

from alembic import op
import sqlalchemy as sa

def upgrade():
    # Step 1: Add new columns as NULLABLE first (safe)
    op.add_column("user", sa.Column("first_name", sa.String(80), nullable=True))
    op.add_column("user", sa.Column("last_name",  sa.String(80), nullable=True))

    # Step 2: DATA MIGRATION — populate new columns from old data
    conn = op.get_bind()
    rows = conn.execute(sa.text("SELECT id, full_name FROM user")).fetchall()

    for row_id, full_name in rows:
        if full_name:
            parts  = (full_name or "").split(" ", 1)
            first  = parts[0]
            last   = parts[1] if len(parts) > 1 else ""
        else:
            first, last = "", ""

        conn.execute(
            sa.text("UPDATE user SET first_name=:f, last_name=:l WHERE id=:id"),
            {"f": first, "l": last, "id": row_id}
        )

    # Step 3: Make NOT NULL now that all rows have values
    op.alter_column("user", "first_name", nullable=False, server_default="")
    op.alter_column("user", "last_name",  nullable=False, server_default="")

    # Step 4: Drop the old column
    op.drop_column("user", "full_name")


def downgrade():
    op.add_column("user", sa.Column("full_name", sa.String(160), nullable=True))
    conn = op.get_bind()
    # Recombine first + last into full_name
    conn.execute(sa.text(
        "UPDATE user SET full_name = first_name || ' ' || last_name"
    ))
    op.alter_column("user", "full_name", nullable=False)
    op.drop_column("user", "last_name")
    op.drop_column("user", "first_name")
'''
print(data_migration)
print()
print("Key insight: Auto-generate handles SCHEMA changes (add/drop/alter).")
print("Data migrations (moving/transforming row data) need op.get_bind() + raw SQL.")


## 🧪 What If? Experimentation

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

### What If 1: Adding a NOT NULL Column to a Table with Existing Rows?

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:
print('''
SCENARIO: You add a NOT NULL column to a table with existing data

  class User(db.Model):
      ...
      # NEW COLUMN with no default, not nullable:
      account_type = db.Column(db.String(20), nullable=False)

  $ flask db migrate -m "add account type"
  # Generated: op.add_column("user", sa.Column("account_type",
  #                           sa.String(20), nullable=False))

  $ flask db upgrade
  # ERROR: Cannot add NOT NULL column with no default value
  # Existing rows cannot have NULL for account_type!

THE FIX — safe three-step approach:

  def upgrade():
      # Step 1: Add as NULLABLE (existing rows get NULL — safe)
      op.add_column("user", sa.Column("account_type", sa.String(20), nullable=True))

      # Step 2: Fill in a value for all existing rows
      conn = op.get_bind()
      conn.execute(sa.text("UPDATE user SET account_type = 'standard'"))

      # Step 3: Now make it NOT NULL (all rows have a value)
      op.alter_column("user", "account_type", nullable=False)

ALTERNATIVE: Use server_default (simpler, let the DB fill the value)
  op.add_column("user", sa.Column(
      "account_type",
      sa.String(20),
      nullable=False,
      server_default="standard"   # database fills existing rows automatically
  ))
  # After upgrade, remove server_default if you don't want it ongoing.

The server_default approach is usually simpler and recommended.
''')


### What If 2: You Accidentally Delete the `migrations/` Folder?

Production-oriented material makes more sense when you see it as reliability work. The tools and steps here exist to make behavior repeatable, observable, and safer to change.



In [ ]:
print('''
SCENARIO: migrations/ folder is deleted or corrupted

WHAT IS LOST:
  - Complete history of schema changes
  - Ability to downgrade
  - Team members cannot sync their databases
  - CI/CD pipeline breaks

RECOVERY OPTIONS:

Option 1: Restore from git (BEST — this is why we commit migrations/)
  $ git checkout -- migrations/
  The entire migrations/ folder is restored from version control.
  This is why "commit migrations/ to git" is the #1 rule.

Option 2: You have the database but lost the files
  # The database is fine — you just lost the migration history
  $ flask db init                              # recreate the folder

  # Generate a "baseline" migration from current model state
  $ flask db migrate -m "baseline after recovery"
  # Review: this should match your CURRENT schema exactly

  # IMPORTANT: Don't run upgrade (tables already exist)!
  # Instead, stamp the DB as being at this revision:
  $ flask db stamp head
  # This records the revision ID without running any SQL

  # Future "flask db migrate" and "flask db upgrade" work normally

Option 3: Full reset (only if you can lose ALL data)
  $ flask db init
  $ flask db migrate -m "initial migration"
  $ flask db upgrade    # creates tables from scratch (DATA LOST!)

LESSON: Add migrations/ to git in your FIRST commit.
''')


### What If 3: Two Developers Create Migrations at the Same Time?

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:
print('''
SCENARIO: Developer A and Developer B both edit models and run
"flask db migrate" on separate branches simultaneously.

  Branch A creates: revision b1c2d3  (adds bio to User)
                    down_revision = "9b0e1a2c3d4f"

  Branch B creates: revision f5a6b7  (adds Tag model)
                    down_revision = "9b0e1a2c3d4f"  <- SAME parent!

After merging both branches into main:
  The migrations chain now has a FORK (two heads):
  9b0e... -> b1c2... (head 1)
  9b0e... -> f5a6... (head 2)

  $ flask db upgrade
  ERROR: Multiple head revisions!
         Please merge these revisions.

HOW TO RESOLVE:

  # Create a merge migration that unifies the two heads
  $ flask db merge -m "merge bio and tag migrations" b1c2d3 f5a6b7

  # This creates a new migration file:
  #   revision      = "zz_merged"
  #   down_revision = ("b1c2d3", "f5a6b7")   # tuple = merge point
  #
  #   def upgrade(): pass   # merge commit does nothing
  #   def downgrade(): pass

  $ flask db upgrade    # applies all three: b1c2, f5a6, then the merge

  $ flask db history
  9b0e... -> b1c2... (merged)    add bio to user
  9b0e... -> f5a6... (merged)    add tag model
  (b1c2, f5a6) -> zz_merged (head)   merge bio and tag migrations

HOW TO PREVENT:
  - Communicate schema changes with your team before starting
  - Keep schema changes in separate short-lived feature branches
  - Rebase feature branches onto main BEFORE running flask db migrate
  - One team member owns schema changes during a sprint
''')


## 🚀 Real-World Project Link

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

Every schema change to our **Blog** is a tracked migration. The initial migration creates all four tables (User, Post, Comment, Tag + post_tags association). Adding `last_login` to User is one migration. Adding a `slug` field to Post is another. Adding a full-text search index is yet another. The entire schema evolution is auditable, reversible, and safe to deploy to production without ever losing user data or blog content.

## 📋 Chapter Summary & Cheat Sheet

In [ ]:
lines = [
    "# ============================================================",
    "#    CHAPTER 7 CHEAT SHEET — DATABASE MIGRATIONS",
    "# ============================================================",
    "",
    "# SETUP",
    "# pip install flask-migrate",
    "from flask_migrate import Migrate",
    "migrate = Migrate(app, db)    # after: db = SQLAlchemy(app)",
    "",
    "# ONE-TIME INITIALIZATION",
    "# $ flask db init",
    "# Commit the migrations/ folder to git immediately!",
    "",
    "# EVERYDAY WORKFLOW",
    "# 1. Edit your model in models.py",
    '# 2. $ flask db migrate -m "describe the change"',
    "# 3. Review migrations/versions/latest_file.py",
    "# 4. $ flask db upgrade",
    "",
    "# KEY COMMANDS",
    "# flask db init               one-time: create migrations/ folder",
    "# flask db migrate -m 'msg'   generate migration from model diff",
    "# flask db upgrade            apply pending migrations",
    "# flask db downgrade          revert most recent migration",
    "# flask db history            show all migrations",
    "# flask db current            show current database revision",
    "# flask db stamp head         mark DB as up-to-date (no SQL run)",
    "# flask db merge r1 r2        resolve parallel migration heads",
    "",
    "# COMMON OPERATIONS IN MIGRATION FILES",
    "from alembic import op",
    "import sqlalchemy as sa",
    "",
    "# Add column (nullable first for existing data!)",
    "op.add_column('user', sa.Column('bio', sa.String(500), nullable=True))",
    "# Make it NOT NULL after populating data:",
    "op.alter_column('user', 'bio', nullable=False)",
    "",
    "# Drop a column",
    "op.drop_column('user', 'bio')",
    "",
    "# Create a new table",
    "op.create_table('tag', sa.Column('id', sa.Integer, primary_key=True),",
    "                sa.Column('name', sa.String(50), unique=True, nullable=False))",
    "",
    "# Data migration (raw SQL)",
    "conn = op.get_bind()",
    "conn.execute(sa.text('UPDATE user SET role = :r WHERE role IS NULL'), {'r': 'user'})",
    "",
    "# NOT NULL SAFE PATTERN",
    "op.add_column('user', sa.Column('role', sa.String(20), nullable=True))",
    "op.get_bind().execute(sa.text("UPDATE user SET role='user' WHERE role IS NULL"))",
    "op.alter_column('user', 'role', nullable=False)",
]
for line in lines:
    print(line)


### Multi-DB and Environment-Specific Migrations

Setup steps are easier to remember when you know what each one unlocks. Read this section as a map from each command, option, or file to the problem it solves, so later errors are easier to diagnose instead of feeling random.



In [ ]:
# Handling different environments with migrations
env_config = '''
# config.py — different database URIs per environment
import os

class Config:
    SECRET_KEY = os.environ.get("SECRET_KEY", "dev-secret")
    SQLALCHEMY_TRACK_MODIFICATIONS = False

class DevelopmentConfig(Config):
    SQLALCHEMY_DATABASE_URI = "sqlite:///dev.db"
    DEBUG = True

class TestingConfig(Config):
    SQLALCHEMY_DATABASE_URI = "sqlite:///:memory:"   # in-memory, fast tests
    TESTING = True
    WTF_CSRF_ENABLED = False

class ProductionConfig(Config):
    SQLALCHEMY_DATABASE_URI = os.environ["DATABASE_URL"]   # from env!


# app.py
config_map = {
    "development": DevelopmentConfig,
    "testing":     TestingConfig,
    "production":  ProductionConfig,
}
env = os.environ.get("FLASK_ENV", "development")
app.config.from_object(config_map[env])

# Running migrations per environment:
# $ FLASK_ENV=development flask db upgrade   <- local SQLite
# $ FLASK_ENV=production  flask db upgrade   <- production PostgreSQL

# Flask-Migrate uses the DATABASE_URI from your app config,
# so the same migration files work across all environments.
'''
print(env_config)


### Checking Migration Status in CI/CD

The important beginner shift here is moving from “it works on my machine” to “it behaves predictably in a real environment.” Each step matters because it reduces surprises when the software runs elsewhere.



In [ ]:
# CI/CD pipeline integration
cicd_code = '''
# In your CI/CD pipeline (GitHub Actions, GitLab CI, etc.):
#
# 1. Run tests with a fresh database:
#    - Use sqlite:///:memory: OR a test PostgreSQL
#    - Run db.create_all() (not migrations) for speed
#
# 2. Check for pending migrations before deploy:
#    flask db check   <- raises error if migrations are pending
#
# 3. Apply migrations as part of deployment:
#    flask db upgrade
#
# GitHub Actions example:
# .github/workflows/deploy.yml:
#
# - name: Run database migrations
#   env:
#     DATABASE_URL: ${{ secrets.DATABASE_URL }}
#   run: |
#     flask db upgrade
#
# - name: Restart app
#   run: |
#     systemctl restart myapp

# Useful migration verification commands:
check_commands = [
    "flask db check       # verify all migrations are applied (exit 1 if not)",
    "flask db current     # show current DB version",
    "flask db heads       # show latest available version",
    "flask db history     # full migration history",
]
for cmd in check_commands:
    print(f"  $ {cmd}")
'''
print(cicd_code)


### A Complete Real-World Migration History

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:
# What a real project's migration history looks like
history_example = '''
# $ flask db history (reading oldest to newest)
#
# <base>        -> a1b2c3d4  (initial migration)
#   Created tables: user, post, comment, tag, post_tags
#
# a1b2c3d4      -> b2c3d4e5  (add user profile fields)
#   op.add_column("user", bio)
#   op.add_column("user", profile_picture_url)
#
# b2c3d4e5      -> c3d4e5f6  (add post view counter)
#   op.add_column("post", view_count INTEGER DEFAULT 0)
#
# c3d4e5f6      -> d4e5f6a7  (add email verification)
#   op.add_column("user", is_email_verified BOOLEAN DEFAULT FALSE)
#   op.add_column("user", email_verification_token VARCHAR(64))
#
# d4e5f6a7      -> e5f6a7b8  (add post slugs for SEO URLs)
#   op.add_column("post", slug VARCHAR(200) UNIQUE)
#   op.execute("UPDATE post SET slug = LOWER(REPLACE(title, " ", "-"))")
#   op.alter_column("post", "slug", nullable=False)
#
# e5f6a7b8      -> f6a7b8c9  (add full-text search index)
#   op.create_index("ix_post_body_fts", "post", ["body"], postgresql_using="gin")
#
# f6a7b8c9      -> g7a8b9c0  (add user follower relationship)
#   op.create_table("followers",
#       sa.Column("follower_id", sa.Integer, sa.ForeignKey("user.id")),
#       sa.Column("followed_id", sa.Integer, sa.ForeignKey("user.id")),
#   )
#
# This is 7 migrations. A mature app might have 50-200 migrations.
# Each one is a safe, reversible, documented schema change.
'''
print(history_example)
print()
print("Key observation: migrations grow incrementally.")
print("  Each migration is small and focused on one change.")
print("  This makes them easy to review, test, and roll back.")


### Alembic Configuration Deep Dive

Setup steps are easier to remember when you know what each one unlocks. Read this section as a map from each command, option, or file to the problem it solves, so later errors are easier to diagnose instead of feeling random.



In [ ]:
# The migrations/env.py file controls how Alembic connects and runs
env_py_overview = '''
# migrations/env.py (key parts — auto-generated, rarely needs editing)

from flask import current_app

# Get the database URL from your Flask app config (NOT hardcoded)
def get_engine():
    return current_app.extensions["migrate"].db.get_engine()

def get_metadata():
    return current_app.extensions["migrate"].db.metadata

def run_migrations_online():
    connectable = get_engine()
    with connectable.connect() as connection:
        context.configure(
            connection=connection,
            target_metadata=get_metadata(),
            # These control what auto-generate detects:
            compare_type=True,          # detect column type changes
            compare_server_default=True # detect default value changes
        )
        with context.begin_transaction():
            context.run_migrations()

# alembic.ini — minimal config (usually don't edit this)
# [alembic]
# script_location = migrations
# sqlalchemy.url  = (overridden by env.py to use Flask config)
'''
print(env_py_overview)

print()
print("When DOES auto-generate miss changes?")
misses = [
    "Column renames (seen as drop + add = DATA LOSS!)",
    "Table renames",
    "Stored procedures, views, triggers",
    "Some complex constraint changes",
    "Server defaults (only if compare_server_default=True)",
    "Custom types not in SQLAlchemy",
]
for m in misses:
    print(f"  - {m}")
print()
print("Always review generated migrations before running upgrade!")


## 💪 Practice Prompts

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

**Challenge 1 — Full Migration Lifecycle:**
Start with a `User` model with only `id`, `username`, and `email`. Run `flask db init` then `flask db migrate -m "initial migration"` then `flask db upgrade`. Now add four new fields: `bio` (String 500, nullable), `profile_picture` (String 300, nullable), `is_verified` (Boolean, default False), and `last_login` (DateTime, nullable). Run `flask db migrate -m "add profile fields"`, review the file, run `flask db upgrade`. Test `flask db downgrade` then `flask db upgrade` again.

**Challenge 2 — NOT NULL Safe Migration:**
Write a migration manually (don't use auto-generate) that adds a `role` column (String 20, NOT NULL) to an existing `User` table that already has rows. Use the three-step approach: (1) add as nullable, (2) `op.get_bind().execute()` to set all rows to 'user', (3) `op.alter_column` to add NOT NULL. Verify it applies without error on a table with pre-existing data.

**Challenge 3 — Simulate a Merge Conflict:**
Create two git branches. In branch A, add `bio` to User. In branch B, add a `Tag` model. Generate a migration in each branch. Merge both to main. You'll have two heads. Run `flask db merge` to create a merge migration. Verify `flask db upgrade` successfully applies all migrations and `flask db history` shows the correct branching structure.


<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='6.%20databases_with_flask_sqlalchemy.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 6: Databases</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../3.%20application_structure/8.%20blueprints_and_application_factory.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 8: Blueprints →</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='6. databases_with_flask_sqlalchemy.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='../3. application_structure/8. blueprints_and_application_factory.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>
